In [1]:

import sys
import logging
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL

import pandas as pd
import numpy as np
from QGI.neoapi import Neo4j

In [2]:
def get_logger():
    logger=logging.getLogger('logger')
    logger.setLevel(logging.INFO)
    #fh=logging.FileHandler("E:/wangzhilin/QuantumGalaxy/logs/aura_api_log.log",'a',encoding='utf-8')
    #fh.setLevel(logging.INFO)
    ch=logging.StreamHandler()
    ch.setLevel(logging.DEBUG)
    formatter=logging.Formatter(fmt='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',datefmt='%m-%d %H:%M')
    ch.setFormatter(formatter)
    #fh.setFormatter(formatter)
    logger.addHandler(ch)
    #logger.addHandler(fh)
    return logger
logger=logging.getLogger()

In [9]:
uri = "neo4j+ssc://08ef0a79.databases.neo4j.io"
user = "QG_Editor"
password = "editor"


neo=Neo4j(uri,user,password)
host='localhost'
user='Local_Editor'
password='QuantumGalaxy'
database='qgdbs'
mysql=MYSQL(host,user,password,database)

In [10]:
uri2='neo4j+ssc://bd2c184f.databases.neo4j.io'
user2='neo4j'
password2='OkvWgegnLgUSuAPl-Zcu8wkcXCX-GxNRC2zydwpbwrE'
neo2=Neo4j(uri2,user2,password2)

In [5]:
mysql.close()
mysql=MYSQL(host,user,password,database)

In [16]:
def neo2mysql():
    cypher='match (n) return id(n),labels(n),n.name,n.code'
    result=neo.read_query(cypher)
    val=[]
    for r in result:
        val.append((str(r.values()[1])[1:-1],r.values()[2],r.values()[3],r.values()[0]))
    val1=[]
    for v in val:
        val1.append(v+(v[0],v[1],v[2]))

    name_val=[]
    for v in val:
        name_val.append((v[3],'name',v[1],v[1]))
    code_val=[]
    for v in val:
        code_val.append((v[3],'code',v[2],v[2]))
    sql='insert into nodes (labels,name,code,id,using_flag) values(%s,%s,%s,%s,1) on duplicate key update labels=%s,name=%s,code=%s,using_flag=1'
    mysql.write_many_query(sql,val1)

    sql2='insert into nodes_records (id,field,record_index,conclusion) values(%s,%s,0,%s) on duplicate key update conclusion=%s'
    mysql.write_many_query(sql2,name_val)
    mysql.write_many_query(sql2,code_val)

    mysql.write_query('SET FOREIGN_KEY_CHECKS = 0')
    mysql.write_query('delete from edges')
    mysql.write_query('SET FOREIGN_KEY_CHECKS = 1')
    cypher='match()-[r]->() return id(startNode(r)),id(endNode(r)),type(r),id(r)  '
    result=neo.read_query(cypher)
    val=[]
    for r in result:
        val.append((str(r.values()[0]),r.values()[1],r.values()[2],r.values()[3],r.values()[2],r.values()[3]))
    sql='insert into edges(start,end,relationship,id,using_flag) value(%s,%s,%s,%s,1) on duplicate key update relationship=%s,id=%s,using_flag=1'
    mysql.write_many_query(sql,val)



In [21]:
neo2mysql()

In [29]:
#result[0]
for r in result:
    id=r[0]
    name=r[2]
    code=r[3]
    if code:
        if name:
            cypher=('match (n)  where id(n)=%d set n.name="%s",n.code="%s"'%(id,name,code))
        else:
            cypher=('match (n) where id(n)=%d set n.code="%s"'%(id,code))
    else:
        if name:
            cypher=('match (n)  where id(n)=%d set n.name="%s"'%(id,name))
        else:
            pass
    neo2.simple_query(cypher)

In [6]:
val=[]
for r in result:
    val.append((str(r.values()[1])[1:-1],r.values()[2],r.values()[3],r.values()[0]))
val

[("'Demand'", '电力需求', None, 0),
 ("'Company', 'Indicator'", '中环装备', '300140.SZ', 1),
 ("'Company', 'Indicator'", '绿洲石油', 'OAS.O', 2),
 ("'Company'", 'CEQP', None, 3),
 ("'Company'", 'phillips66', None, 4),
 ("'Product'", '二硫化钼', None, 5),
 ("'Product'", '风力发电站', None, 6),
 ("'Product'", '变压器组件', None, 7),
 ("'Scenario'", 'JKS.N 晶科能源 情景 on 2022-08-01', 'JKS.N', 8),
 ("'Company', 'Indicator'", '兰花科创', '600123.SH', 9),
 ("'Demand'", '政府类需求', None, 10),
 ("'Product'", '己内酰胺', None, 11),
 ("'Product'", '棕榈油\n', None, 12),
 ("'Product'", '测试产品', None, 13),
 ("'Product'", '纯苯', None, 14),
 ("'Product'", '合成氨', None, 15),
 ("'Company'", '晋城市益兴达工贸有限公司', None, 16),
 ("'Product'", '压裂服务', None, 17),
 ("'Product'", '尼龙原料', None, 18),
 ("'Demand'", '企业用房需求', None, 19),
 ("'Product'", '棕榈树', None, 20),
 ("'Product'", '甲醇', None, 21),
 ("'Product'", '二甲醚', None, 22),
 ("'Demand'", '民用燃料需求', None, 23),
 ("'Company'", '东平百成矿业有限公司', None, 24),
 ("'Product'", '油棕种子', None, 25),
 ("'Technique'", '风力发电', N

In [15]:
val1=[]
for v in val:
    val1.append(v+(v[0],v[1],v[2]))
val1
name_val=[]
for v in val:
    name_val.append((v[3],'name',v[1],v[1]))
code_val=[]
for v in val:
    code_val.append((v[3],'code',v[2],v[2]))
code_val

[(0, 'code', None, None),
 (1, 'code', '300140.SZ', '300140.SZ'),
 (2, 'code', 'OAS.O', 'OAS.O'),
 (3, 'code', None, None),
 (4, 'code', None, None),
 (5, 'code', None, None),
 (6, 'code', None, None),
 (7, 'code', None, None),
 (8, 'code', 'JKS.N', 'JKS.N'),
 (9, 'code', '600123.SH', '600123.SH'),
 (10, 'code', None, None),
 (11, 'code', None, None),
 (12, 'code', None, None),
 (13, 'code', None, None),
 (14, 'code', None, None),
 (15, 'code', None, None),
 (16, 'code', None, None),
 (17, 'code', None, None),
 (18, 'code', None, None),
 (19, 'code', None, None),
 (20, 'code', None, None),
 (21, 'code', None, None),
 (22, 'code', None, None),
 (23, 'code', None, None),
 (24, 'code', None, None),
 (25, 'code', None, None),
 (26, 'code', None, None),
 (27, 'code', None, None),
 (28, 'code', '3668.HK', '3668.HK'),
 (29, 'code', None, None),
 (30, 'code', None, None),
 (31, 'code', None, None),
 (32, 'code', None, None),
 (33, 'code', None, None),
 (34, 'code', None, None),
 (35, 'code', '

In [8]:
len(val1)

4461

In [20]:
sql='insert into nodes (labels,name,code,id,using_flag) values(%s,%s,%s,%s,1) on duplicate key update labels=%s,name=%s,code=%s,using_flag=1'
mysql.write_many_query(sql,val1)

sql2='insert into nodes_records (id,field,record_index,conclusion) values(%s,%s,0,%s) on duplicate key update conclusion=%s'
mysql.write_many_query(sql2,name_val)


4459

In [21]:

mysql.write_many_query(sql2,code_val)

4460

In [42]:
cypher='match()-[r]->() return id(startNode(r)),id(endNode(r)),type(r),id(r)  '
result=neo.read_query(cypher)
result

[<Record id(startNode(r))=0 id(endNode(r))=1725 type(r)='parent' id(r)=2646>,
 <Record id(startNode(r))=0 id(endNode(r))=88 type(r)='parent' id(r)=1696>,
 <Record id(startNode(r))=0 id(endNode(r))=378 type(r)='parent' id(r)=1695>,
 <Record id(startNode(r))=1 id(endNode(r))=1575 type(r)='supply' id(r)=2091>,
 <Record id(startNode(r))=1 id(endNode(r))=1579 type(r)='produce' id(r)=2088>,
 <Record id(startNode(r))=1 id(endNode(r))=1577 type(r)='produce' id(r)=2085>,
 <Record id(startNode(r))=1 id(endNode(r))=1576 type(r)='produce' id(r)=2084>,
 <Record id(startNode(r))=1 id(endNode(r))=1573 type(r)='produce' id(r)=1873>,
 <Record id(startNode(r))=1 id(endNode(r))=882 type(r)='produce' id(r)=1869>,
 <Record id(startNode(r))=1 id(endNode(r))=7 type(r)='produce' id(r)=1270>,
 <Record id(startNode(r))=1 id(endNode(r))=1572 type(r)='supply' id(r)=1868>,
 <Record id(startNode(r))=1 id(endNode(r))=1571 type(r)='supply' id(r)=1000>,
 <Record id(startNode(r))=2 id(endNode(r))=4 type(r)='supply' id(

In [44]:
len(result)

6956

In [47]:
val=[]
for r in result:
    val.append((str(r.values()[0]),r.values()[1],r.values()[2],r.values()[3],r.values()[2],r.values()[3]))
sql='insert into edges(start,end,relationship,id,using_flag) value(%s,%s,%s,%s,1) on duplicate key update relationship=%s,id=%s,using_flag=1'
mysql.write_many_query(sql,val)

0

In [30]:

sql='update nodes set labels=%s ,name=%s,using_flag=1 where id =%s'
r=mysql.write_many_query(sql,val)
r

Had problem on sql "update nodes set labels=%s ,name=%s,using_flag=1 where id =%s" and rollback
not all arguments converted during string formatting


In [ ]:
def refresh_flag():
    
    cypher='match (n) return id(n)'
    r=neo.simple_query(cypher)
    using=[]
    for record in r:
        using.append((record.value(),))
    sql0='update nodes set using_flag=0'
    sql='update nodes set using_flag=1 where id=%s'
    mysql.write_query(sql0)
    n=mysql.write_many_query(sql,using)
    print(n)
    
    cypher='match ()-[r]->() return id(r)'
    r=neo.simple_query(cypher)
    using=[]
    for record in r:
        using.append((record.value(),))
    sql0='update edges set using_flag=0'
    sql='update edges set using_flag=1 where id=%s'
    mysql.write_query(sql0)
    e=mysql.write_many_query(sql,using)
    print(e)